# Models, messages and tools

A chat model produces messages. An agent adds the tool-execution loop: a model requests a named function with arguments, Python executes it, and the result becomes another message.

[Model and tool execution](https://docs.langchain.com/oss/python/langchain/agents)

In [ ]:
# Imports and project path setup only; examples start in the next cell.
from pathlib import Path
import json
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
LESSONS = ROOT / "notebooks" / "langchain" / "06_middlewares"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(LESSONS) not in sys.path:
    sys.path.insert(0, str(LESSONS))

from lab.config import profile_for, settings
from lab.data import conversation, prompt
from lab.display import TraceMiddleware, expect_error, show_messages, show_source
from lab.tools import add, broken_inventory, inventory, make_flaky_tool, next_page, poll_job
from langchain.agents import create_agent
from langchain_core.utils.function_calling import convert_to_openai_tool
from model_config import get_model, run_config

In [ ]:
# load the model
model = get_model()

In [ ]:
print(prompt("hello"))
print(model.invoke(prompt("hello")).content)

### Conversation history and tool schemas

A conversation contains ordered human, assistant and tool messages. A tool schema describes a function name, argument types and purpose so the model can request a valid call.

In [ ]:
# Execute
for name in ("project", "incident", "hiring"):
    messages = conversation(name)
    print(name, "messages:", len(messages), "characters:", sum(len(m.content) for m in messages))

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(conversation("project")[:4], pretty=True)

In [ ]:
print("Available tools:", [tool.name for tool in inventory()])
# Inspect a full schema later with convert_to_openai_tool(inventory()[0]).


### The model/tool cycle

The model requests `add`; Python executes it and returns a tool message. The model then uses that result to produce its answer. Requests, tool calls and results are observable; internal model reasoning is not part of this execution history.

In [ ]:
# Execute
trace = TraceMiddleware("00_baseline", pretty=True)
agent = create_agent(model=model, tools=[add], middleware=[trace])
result = agent.invoke({"messages": [("user", prompt("add"))]}, config=run_config())

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(result["messages"], pretty=True)
print("Saved log:", trace.path)

### Permanent errors, transient errors and nontermination

An implementation error requires a code fix. A transient connection failure can recover on retry. Polling and pagination can return valid results forever without reaching completion, whereas a blocked worker needs a time limit.

In [ ]:
show_source(broken_inventory)
with expect_error(TypeError):
    broken_inventory.invoke({"product": "keyboard"})
print(poll_job.invoke({"job_id": "DEMO-42"}))
print(next_page.invoke({"cursor": 0}))